In [2]:
# =========================================
# Step RR-1 · Candidate Union (K_cand=200)  —  Patched manifest parser
# =========================================
import os, json, gc
from pathlib import Path
import numpy as np
import pandas as pd

PARQUET_ENGINE = "fastparquet"
TMP_DIR = Path("./tmp").resolve()

# ---- 参数 ----
K_PER_VIEW   = 50           # 输入每图的K
K_CAND       = 200          # 候选上限
ROWS_PER_PART= 10_000       # 每片行数
SEED         = 123

# 输入前缀（对称化 + 行归一 + k=50）
PREF_TAG   = "S_tag_symrow_k50"
PREF_TEXT  = "S_text_symrow_k50"
PREF_BEH   = "S_beh_symrow_k50"
PREF_FUSE  = "S_fused3_symrow_k50"

# 输出前缀
OUT_PREF   = "S_fused3_rr_cand_k200"

np.random.seed(SEED)
print(f"[Env] TMP_DIR = {TMP_DIR}")

# ---------- 工具：稳健的类型转换 ----------
def _to_int(x, default=None):
    """把 x 转成 int；支持 str(含逗号)、float、list(取 len 或 sum)"""
    if x is None:
        return default
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, float):
        return int(x)
    if isinstance(x, str):
        s = x.replace(",", "").replace("_","").strip()
        try:
            return int(float(s))
        except:
            return default
    if isinstance(x, list):
        # 常见两种：parts 为 list → 用 len；nnz 为 list → 用 sum
        # 但这里不知语义，交由调用处决定；这里返回 None 让上层处理
        return None
    return default

def read_manifest(prefix: str):
    man_path = TMP_DIR / f"{prefix}_manifest.json"
    if not man_path.exists():
        # 兼容早期命名
        cands = sorted(TMP_DIR.glob(f"{prefix}*_manifest.json"))
        if not cands:
            raise FileNotFoundError(f"[FATAL] manifest not found for prefix={prefix}")
        man_path = cands[0]

    with open(man_path, "r") as f:
        man = json.load(f)

    # nodes
    nodes = man.get("nodes", man.get("N", man.get("n", None)))
    n_val = _to_int(nodes)
    if n_val is None:
        # 尝试从 index_map 推断
        idmap_path = TMP_DIR / "index_map.parquet"
        if idmap_path.exists():
            id_map = pd.read_parquet(idmap_path, engine=PARQUET_ENGINE, columns=["doc_idx"])
            n_val = int(id_map["doc_idx"].max()) + 1
        else:
            # 最后兜底：从某个 part 文件中取最大 row/col
            part_files = sorted(TMP_DIR.glob(f"{prefix}_part*.parquet"))
            if not part_files:
                raise FileNotFoundError(f"[FATAL] cannot infer nodes, no part files for {prefix}")
            mx = 0
            for fp in part_files[:3]:
                df = pd.read_parquet(fp, engine=PARQUET_ENGINE, columns=["row","col"])
                mx = max(mx, int(df["row"].max()), int(df["col"].max()))
            n_val = mx + 1

    # parts
    parts = man.get("parts", None)
    p_val = _to_int(parts)
    files = man.get("files", None)
    if p_val is None:
        if isinstance(parts, list):
            p_val = len(parts)
        elif isinstance(files, list):
            p_val = len(files)
        else:
            # 扫描磁盘兜底
            p_val = len(sorted(TMP_DIR.glob(f"{prefix}_part*.parquet")))
            if p_val == 0:
                raise FileNotFoundError(f"[FATAL] no part files for {prefix}")

    # nnz（仅打印用，可为 None；若是列表则取 sum）
    nnz = man.get("nnz", man.get("total_edges", None))
    nnz_val = _to_int(nnz)
    if nnz_val is None and isinstance(nnz, list):
        try:
            nnz_val = int(sum([_to_int(x, 0) for x in nnz]))
        except:
            nnz_val = None

    # files（可选）
    if isinstance(files, list) and files:
        file_list = [str((TMP_DIR / fn).name if not str(fn).startswith(str(TMP_DIR)) else Path(fn).name)
                     for fn in files]
    else:
        file_list = []

    return {"nodes": int(n_val), "parts": int(p_val), "nnz": nnz_val, "files": file_list}

def list_part_files(prefix: str, manifest: dict):
    # 优先使用 manifest.files
    if manifest.get("files"):
        fps = [TMP_DIR / fn for fn in manifest["files"]]
        fps = [fp for fp in fps if fp.exists()]
        if fps:
            return fps
    # 否则按规则扫描
    fps = []
    for p in range(manifest["parts"]):
        fp = TMP_DIR / f"{prefix}_part{p:04d}.parquet"
        if fp.exists():
            fps.append(fp)
    if not fps:
        fps = sorted(TMP_DIR.glob(f"{prefix}_part*.parquet"))
    if not fps:
        raise FileNotFoundError(f"[FATAL] no part files for prefix={prefix}")
    return fps

# ---------- 两遍读构建 CSR ----------
def build_csr(prefix: str):
    man = read_manifest(prefix)
    N = man["nodes"]
    part_files = list_part_files(prefix, man)
    print(f"[LOAD] {prefix}: nodes={N:,}, parts={len(part_files)}, nnz≈{man['nnz'] if man['nnz'] else 'unknown'}")

    # pass-1: 计数
    counts = np.zeros(N, dtype=np.int64)
    for i, fp in enumerate(part_files, 1):
        df = pd.read_parquet(fp, engine=PARQUET_ENGINE, columns=["row"])
        r = df["row"].to_numpy(dtype=np.int64, copy=False)
        np.add.at(counts, r, 1)
        if i % 4 == 0 or i == len(part_files):
            print(f"[PASS1] {prefix} parts {i}/{len(part_files)} counted")
        del df
    indptr = np.zeros(N + 1, dtype=np.int64)
    np.cumsum(counts, out=indptr[1:])
    nnz = int(indptr[-1])
    indices = np.empty(nnz, dtype=np.int32)
    values  = np.empty(nnz, dtype=np.float32)
    write_ptr = indptr[:-1].copy()

    # pass-2: 填充
    for i, fp in enumerate(part_files, 1):
        df = pd.read_parquet(fp, engine=PARQUET_ENGINE, columns=["row","col","val"])
        rows = df["row"].to_numpy(np.int64,  copy=False)
        cols = df["col"].to_numpy(np.int32,  copy=False)
        vals = df["val"].to_numpy(np.float32, copy=False)
        order = np.argsort(rows, kind="stable")
        rows, cols, vals = rows[order], cols[order], vals[order]
        if len(rows) > 0:
            uniq_rows, start_idx = np.unique(rows, return_index=True)
            start_idx = np.r_[start_idx, len(rows)]
            for k in range(len(uniq_rows)):
                r = int(uniq_rows[k]); s, e = int(start_idx[k]), int(start_idx[k+1])
                wpos = write_ptr[r]
                ln = e - s
                indices[wpos:wpos+ln] = cols[s:e]
                values [wpos:wpos+ln] = vals[s:e]
                write_ptr[r] += ln
        if i % 4 == 0 or i == len(part_files):
            print(f"[PASS2] {prefix} parts {i}/{len(part_files)} filled")
        del df, rows, cols, vals, order
        gc.collect()
    return N, indptr, indices, values

# ---------- 读取四图为 CSR ----------
N_tag,  ip_tag,  ix_tag,  vx_tag  = build_csr(PREF_TAG)
N_text, ip_text, ix_text, vx_text = build_csr(PREF_TEXT)
N_beh,  ip_beh,  ix_beh,  vx_beh  = build_csr(PREF_BEH)
N_fus,  ip_fus,  ix_fus,  vx_fus  = build_csr(PREF_FUSE)
assert N_tag == N_text == N_beh == N_fus, "[FATAL] node size mismatch among views"
N = N_tag
print(f"[CSR] all views loaded. N={N:,}")

# ---------- 并集 + 标注 + 写分片 ----------
out_parts = 0
rows = np.arange(N, dtype=np.int64)
part_bounds = np.arange(0, N, ROWS_PER_PART, dtype=np.int64)
if part_bounds[-1] != N:
    part_bounds = np.r_[part_bounds, N]

out_files = []
for bi in range(len(part_bounds)-1):
    rs, re = int(part_bounds[bi]), int(part_bounds[bi+1])
    recs = []

    for r in range(rs, re):
        a = ix_tag [ip_tag [r]:ip_tag [r+1]]
        b = ix_text[ip_text[r]:ip_text[r+1]]
        c = ix_beh [ip_beh [r]:ip_beh [r+1]]
        d = ix_fus [ip_fus [r]:ip_fus [r+1]]

        cand = np.unique(np.concatenate((a,b,c,d)))
        if cand.size > K_CAND:
            dv = ix_fus[ip_fus[r]:ip_fus[r+1]]
            dw = vx_fus[ip_fus[r]:ip_fus[r+1]]
            base_score = np.zeros(cand.size, dtype=np.float32)
            if dv.size:
                # 简单线性查找即可；数组很小（<=200）
                for i, cj in enumerate(cand):
                    hit = np.where(dv == cj)[0]
                    base_score[i] = dw[int(hit[0])] if hit.size else 0.0
            take = np.argsort(-base_score, kind="stable")[:K_CAND]
            cand = cand[take]

        aset = set(a.tolist())
        bset = set(b.tolist())
        cset = set(c.tolist())
        dv   = ix_fus[ip_fus[r]:ip_fus[r+1]]
        dw   = vx_fus[ip_fus[r]:ip_fus[r+1]]
        fus_map = {int(dv[i]): float(dw[i]) for i in range(len(dv))}

        for j in cand:
            recs.append((
                int(r), int(j),
                1 if j in aset else 0,
                1 if j in bset else 0,
                1 if j in cset else 0,
                float(fus_map.get(int(j), 0.0))
            ))

    out_df = pd.DataFrame.from_records(
        recs, columns=["row","col","in_tag","in_text","in_beh","fused_base"]
    )
    out_fp = TMP_DIR / f"{OUT_PREF}_part{out_parts:04d}.parquet"
    out_df.to_parquet(out_fp, engine=PARQUET_ENGINE, index=False)
    out_files.append(str(out_fp.name))
    out_parts += 1
    print(f"[SAVE] part#{out_parts-1} rows={len(out_df):,} -> {out_fp.name}")
    del out_df, recs
    gc.collect()

man = {
    "nodes": int(N),
    "K_cand": int(K_CAND),
    "parts": int(out_parts),
    "source": {"tag": PREF_TAG, "text": PREF_TEXT, "beh": PREF_BEH, "fused": PREF_FUSE},
    "files": out_files
}
with open(TMP_DIR / f"{OUT_PREF}_manifest.json", "w") as f:
    json.dump(man, f)
print(f"[MANIFEST] {OUT_PREF}_manifest.json written. parts={out_parts}")
print("[STEP RR-1] DONE: candidate union built.")


[Env] TMP_DIR = /home/koyo/workspace/recsys/tmp
[LOAD] S_tag_symrow_k50: nodes=521,735, parts=15, nnz≈28159756
[PASS1] S_tag_symrow_k50 parts 4/15 counted
[PASS1] S_tag_symrow_k50 parts 8/15 counted
[PASS1] S_tag_symrow_k50 parts 12/15 counted
[PASS1] S_tag_symrow_k50 parts 15/15 counted
[PASS2] S_tag_symrow_k50 parts 4/15 filled
[PASS2] S_tag_symrow_k50 parts 8/15 filled
[PASS2] S_tag_symrow_k50 parts 12/15 filled
[PASS2] S_tag_symrow_k50 parts 15/15 filled
[LOAD] S_text_symrow_k50: nodes=521,735, parts=15, nnz≈28161106
[PASS1] S_text_symrow_k50 parts 4/15 counted
[PASS1] S_text_symrow_k50 parts 8/15 counted
[PASS1] S_text_symrow_k50 parts 12/15 counted
[PASS1] S_text_symrow_k50 parts 15/15 counted
[PASS2] S_text_symrow_k50 parts 4/15 filled
[PASS2] S_text_symrow_k50 parts 8/15 filled
[PASS2] S_text_symrow_k50 parts 12/15 filled
[PASS2] S_text_symrow_k50 parts 15/15 filled
[LOAD] S_beh_symrow_k50: nodes=521,735, parts=14, nnz≈26023186
[PASS1] S_beh_symrow_k50 parts 4/14 counted
[PASS1

In [1]:
# =========================================
# Step RR-2 · Rerank with support + light boosts + temperature softmax
# 输入:  tmp/S_fused3_rr_cand_k200_part*.parquet (+ manifest)
#        tmp/relevance_tag_docs.parquet, tmp/relevance_tag_idf.parquet
#        tmp/beh_base.parquet
# 输出:  tmp/S_fused3_rr_k50_part*.parquet + manifest
# =========================================
import json, gc
from pathlib import Path
import numpy as np
import pandas as pd

PARQUET_ENGINE = "fastparquet"
TMP_DIR = Path("./tmp").resolve()

# ---------- 可调参数 ----------
K_FINAL   = 50
TAU       = 1.5     # 温度（>1 压缩头部优势）
GAMMA     = 0.15    # support 一致性奖励权重
LAMBDA_TAG= 0.12    # Tag 重合 boost
LAMBDA_BEH= 0.08    # 行为轻吻合 boost
EPS       = 1e-8

CAND_PREF = "S_fused3_rr_cand_k200"  # RR-1 输出
OUT_PREF  = "S_fused3_rr_k50"        # 本步输出

print(f"[Env] TMP_DIR = {TMP_DIR}")

# ---------- 小工具 ----------
def _robust_minmax(x: np.ndarray):
    """按行做稳健 min-max 归一到 [0,1]（使用 5%~95% 分位裁剪，避免极端值）"""
    if x.size == 0:
        return x
    lo = np.quantile(x, 0.05) if x.size >= 20 else x.min()
    hi = np.quantile(x, 0.95) if x.size >= 20 else x.max()
    if hi - lo < 1e-12:
        return np.zeros_like(x, dtype=np.float32)
    x2 = np.clip(x, lo, hi)
    return ((x2 - lo) / (hi - lo)).astype(np.float32)

def _softmax_t(x: np.ndarray, tau: float):
    m = np.max(x)
    z = np.exp((x - m) / max(tau, 1e-6))
    s = z.sum()
    return (z / max(s, 1e-12)).astype(np.float32)

# ---------- 读取候选 manifest ----------
cand_man_path = TMP_DIR / f"{CAND_PREF}_manifest.json"
with open(cand_man_path, "r") as f:
    cand_man = json.load(f)
CAND_PARTS = cand_man.get("parts") or len(cand_man.get("files", []))
N_NODES    = int(cand_man.get("nodes"))
cand_files = cand_man.get("files")
if not cand_files:
    # 兜底：按规则扫描
    cand_files = [f"S_fused3_rr_cand_k200_part{p:04d}.parquet" for p in range(CAND_PARTS)]
print(f"[MAN] candidates: nodes={N_NODES:,}, parts={len(cand_files)}")

# ---------- 加载 Tag 侧缓存 ----------
tag_docs_path = TMP_DIR / "relevance_tag_docs.parquet"
tag_idf_path  = TMP_DIR / "relevance_tag_idf.parquet"
assert tag_docs_path.exists() and tag_idf_path.exists(), "[FATAL] tag relevance cache missing"

tag_docs = pd.read_parquet(tag_docs_path, engine=PARQUET_ENGINE)
# 支持 'tags' 或 'tag_list' 两种列名
if "tags" in tag_docs.columns:
    doc_tags_series = tag_docs.set_index("doc_idx")["tags"]
elif "tag_list" in tag_docs.columns:
    doc_tags_series = tag_docs.set_index("doc_idx")["tag_list"]
else:
    raise KeyError("[FATAL] relevance_tag_docs 缺少 tags/tag_list 列")
# 转为字典（只有有标签的文档会出现；其他默认为空集合）
doc2tags = {int(k): set(v) if isinstance(v, (list, set, tuple)) else set() 
            for k, v in doc_tags_series.items()}

tag_idf = pd.read_parquet(tag_idf_path, engine=PARQUET_ENGINE)
# 寻找 idf 的键列
idf_col = None
for c in ["tag","token","term","word","key"]:
    if c in tag_idf.columns:
        idf_col = c
        break
if idf_col is None:
    # 兼容以 'col' 存整数 id 的情况（不常见）
    if "col" in tag_idf.columns:
        # 这类通常与整数词表相配，此场景我们无法从字符串交集计算，退化为零 boost
        print("[WARN] tag_idf 使用整数索引，无法计算字面交集，将禁用 Tag 重合 boost。")
        LAMBDA_TAG = 0.0
        idf_map = {}
    else:
        raise KeyError("[FATAL] relevance_tag_idf 无法找到 tag 键列")
else:
    idf_map = {str(row[idf_col]): float(row["idf"]) for _, row in tag_idf.iterrows()}

# 为每个文档预存 denom = sum idf(tag)（无则 0）
denom_idf = np.zeros(N_NODES, dtype=np.float32)
for d, ts in doc2tags.items():
    if not ts:
        continue
    s = 0.0
    for t in ts:
        s += float(idf_map.get(str(t), 0.0))
    denom_idf[d] = s

print(f"[TAG] tag docs loaded: {len(doc2tags):,} docs with tags; idf entries={len(idf_map):,}")

# ---------- 加载行为基表 ----------
beh_base = pd.read_parquet(TMP_DIR / "beh_base.parquet", engine=PARQUET_ENGINE)
org_arr = np.full(N_NODES, -1, dtype=np.int64)
cre_arr = np.full(N_NODES, -1, dtype=np.int64)
bdx = beh_base["doc_idx"].to_numpy(np.int64)
if "OwnerOrganizationId" in beh_base.columns:
    orgv = beh_base["OwnerOrganizationId"].fillna(-1).astype("Int64").to_numpy()
else:
    orgv = np.full(bdx.size, -1, dtype="Int64")
if "CreatorUserId" in beh_base.columns:
    crev = beh_base["CreatorUserId"].fillna(-1).astype("Int64").to_numpy()
else:
    crev = np.full(bdx.size, -1, dtype="Int64")
org_arr[bdx] = np.array(orgv, dtype=np.int64)
cre_arr[bdx] = np.array(crev, dtype=np.int64)
print("[BEH] beh_base loaded.")

# ---------- 逐分片重排并落盘 ----------
out_files = []
for pi, fn in enumerate(cand_files, 1):
    fp = TMP_DIR / fn
    df = pd.read_parquet(fp, engine=PARQUET_ENGINE)
    # 期待列: row, col, in_tag, in_text, in_beh, fused_base
    assert {"row","col","in_tag","in_text","in_beh","fused_base"}.issubset(df.columns), \
        f"[FATAL] columns missing in {fn}"

    # 按 row 分组（每组 <= 200 候选）
    rows = df["row"].to_numpy(np.int64, copy=False)
    order = np.argsort(rows, kind="stable")
    rows_sorted = rows[order]
    cols_sorted = df["col"].to_numpy(np.int64, copy=False)[order]
    tagf = df["in_tag"].to_numpy(np.int32, copy=False)[order]
    textf= df["in_text"].to_numpy(np.int32, copy=False)[order]
    behf = df["in_beh"].to_numpy(np.int32, copy=False)[order]
    base = df["fused_base"].to_numpy(np.float32, copy=False)[order]

    uniq_rows, start_idx = np.unique(rows_sorted, return_index=True)
    start_idx = np.r_[start_idx, len(rows_sorted)]

    out_rec = []
    for k in range(len(uniq_rows)):
        r = int(uniq_rows[k])
        s, e = int(start_idx[k]), int(start_idx[k+1])

        cand_cols = cols_sorted[s:e].astype(np.int64, copy=False)
        support   = (tagf[s:e] + textf[s:e] + behf[s:e]).astype(np.int32, copy=False)
        fused_b   = base[s:e].astype(np.float32, copy=False)

        # 1) 一致性奖励
        s_cons = fused_b * (1.0 + GAMMA * np.maximum(0, support - 1))

        # 2) Tag 重合 IDF
        if LAMBDA_TAG > 0.0:
            tags_r = doc2tags.get(r, set())
            denom  = float(denom_idf[r]) + EPS
            st = np.zeros_like(fused_b, dtype=np.float32)
            if tags_r:
                # 逐候选做小集合交集（|tags| 平均≈2，开销极低）
                for i, j in enumerate(cand_cols):
                    tj = doc2tags.get(int(j), set())
                    if not tj:
                        continue
                    inter = tags_r.intersection(tj)
                    if not inter:
                        continue
                    s_idf = 0.0
                    for t in inter:
                        s_idf += float(idf_map.get(str(t), 0.0))
                    st[i] = s_idf / denom
            # 稳健归一
            st = _robust_minmax(st)
        else:
            st = np.zeros_like(fused_b, dtype=np.float32)

        # 3) 行为轻吻合（同 org / 同 creator；-1 视为缺失，不计匹配）
        org_r = org_arr[r]; cre_r = cre_arr[r]
        so = np.zeros_like(fused_b, dtype=np.float32)
        if org_r >= 0 or cre_r >= 0:
            org_j = org_arr[cand_cols]
            cre_j = cre_arr[cand_cols]
            hit_org = ((org_r >= 0) & (org_j >= 0) & (org_j == org_r)).astype(np.float32)
            hit_cre = ((cre_r >= 0) & (cre_j >= 0) & (cre_j == cre_r)).astype(np.float32)
            so = 0.5 * hit_org + 0.5 * hit_cre
        so = _robust_minmax(so)

        # 融合
        s_boost = s_cons + LAMBDA_TAG * st + LAMBDA_BEH * so

        # 温度 softmax & 选 Top-K
        prob = _softmax_t(s_boost, TAU)
        if prob.size > K_FINAL:
            take = np.argsort(-prob, kind="stable")[:K_FINAL]
            sel_c = cand_cols[take]
            sel_v = prob[take]
            # 行内归一，保证随机游走解释概率
            sel_v = (sel_v / max(sel_v.sum(), 1e-12)).astype(np.float32)
        else:
            sel_c = cand_cols
            sel_v = (prob / max(prob.sum(), 1e-12)).astype(np.float32)

        # 记录
        out_rec.extend([(r, int(c), float(v)) for c, v in zip(sel_c, sel_v)])

        if (k % 2000) == 0:
            print(f"[PROG] part {pi}/{len(cand_files)} row {k}/{len(uniq_rows)}")

    # 写分片
    out_df = pd.DataFrame(out_rec, columns=["row","col","val"])
    out_fp = TMP_DIR / f"{OUT_PREF}_part{pi-1:04d}.parquet"
    out_df.to_parquet(out_fp, engine=PARQUET_ENGINE, index=False)
    out_files.append(out_fp.name)
    print(f"[SAVE] {out_fp.name} edges={len(out_df):,}")
    del df, out_df, out_rec
    gc.collect()

# 写 manifest
man = {
    "nodes": int(N_NODES),
    "k": int(K_FINAL),
    "parts": int(len(out_files)),
    "files": out_files,
    "source_candidates": f"{CAND_PREF}_manifest.json",
    "params": {
        "tau": TAU, "gamma": GAMMA, "lambda_tag": LAMBDA_TAG, "lambda_beh": LAMBDA_BEH
    }
}
with open(TMP_DIR / f"{OUT_PREF}_manifest.json", "w") as f:
    json.dump(man, f)
print(f"[MANIFEST] {OUT_PREF}_manifest.json written. parts={len(out_files)}")
print("[STEP RR-2] DONE: reranked graph saved.")


[Env] TMP_DIR = /Users/koyo/workspace/recsys/tmp
[MAN] candidates: nodes=521,735, parts=53
[TAG] tag docs loaded: 521,735 docs with tags; idf entries=394
[BEH] beh_base loaded.
[PROG] part 1/53 row 0/10000
[PROG] part 1/53 row 2000/10000
[PROG] part 1/53 row 4000/10000
[PROG] part 1/53 row 6000/10000
[PROG] part 1/53 row 8000/10000
[SAVE] S_fused3_rr_k50_part0000.parquet edges=500,000
[PROG] part 2/53 row 0/10000
[PROG] part 2/53 row 2000/10000
[PROG] part 2/53 row 4000/10000
[PROG] part 2/53 row 6000/10000
[PROG] part 2/53 row 8000/10000
[SAVE] S_fused3_rr_k50_part0001.parquet edges=500,000
[PROG] part 3/53 row 0/10000
[PROG] part 3/53 row 2000/10000
[PROG] part 3/53 row 4000/10000
[PROG] part 3/53 row 6000/10000
[PROG] part 3/53 row 8000/10000
[SAVE] S_fused3_rr_k50_part0002.parquet edges=500,000
[PROG] part 4/53 row 0/10000
[PROG] part 4/53 row 2000/10000
[PROG] part 4/53 row 4000/10000
[PROG] part 4/53 row 6000/10000
[PROG] part 4/53 row 8000/10000
[SAVE] S_fused3_rr_k50_part0003.p

In [2]:
# =========================================
# Step RR-3 · Evaluate Fused3-RR @K=20 and append to tmp/metrics_main.csv
# 输入:  tmp/S_fused3_rr_k50_manifest.json + 分片
#       tmp/relevance_tag_docs.parquet, tmp/relevance_tag_idf.parquet
#       tmp/beh_base.parquet
# 输出:  在屏幕打印一行 DataFrame；追加写入 tmp/metrics_main.csv
# =========================================
import json, math
from pathlib import Path
import numpy as np
import pandas as pd

PARQUET_ENGINE = "fastparquet"
TMP_DIR = Path("./tmp").resolve()
PREF = "S_fused3_rr_k50"
K_EVAL = 20

W_TAG, W_ORG, W_CRE = 0.6, 0.1, 0.3  # 统一指标权重

print(f"[Env] TMP_DIR={TMP_DIR}")

# ---------- 读取 manifest ----------
man_path = TMP_DIR / f"{PREF}_manifest.json"
with open(man_path, "r") as f:
    man = json.load(f)
N = int(man["nodes"])
parts = man.get("parts") or len(man.get("files", []))
files = man.get("files")
if not files:
    files = [f"{PREF}_part{p:04d}.parquet" for p in range(parts)]
print(f"[MAN] {PREF}: nodes={N:,}, parts={len(files)}")

# ---------- 读取 Tag 银标缓存 ----------
tag_docs = pd.read_parquet(TMP_DIR/"relevance_tag_docs.parquet", engine=PARQUET_ENGINE)
# 支持 tags / tag_list 两种列名
if "tags" in tag_docs.columns:
    doc_tags_series = tag_docs.set_index("doc_idx")["tags"]
elif "tag_list" in tag_docs.columns:
    doc_tags_series = tag_docs.set_index("doc_idx")["tag_list"]
else:
    raise KeyError("[FATAL] relevance_tag_docs 缺少 tags/tag_list 列")
doc2tags = {int(k): (set(v) if isinstance(v, (list,set,tuple)) else set()) for k, v in doc_tags_series.items()}

tag_idf = pd.read_parquet(TMP_DIR/"relevance_tag_idf.parquet", engine=PARQUET_ENGINE)
idf_key = None
for c in ["tag","token","term","word","key"]:
    if c in tag_idf.columns: idf_key=c; break
if idf_key is None:
    raise KeyError("[FATAL] relevance_tag_idf 缺少可识别的键列")
idf_map = {str(r[idf_key]): float(r["idf"]) for _, r in tag_idf.iterrows()}

# 预备：每文档标签 IDF 和；每标签文档计数（用于 Tag-Recall 分母的上界近似）
denom_idf = np.zeros(N, dtype=np.float32)
tag_freq = {}
for d, ts in doc2tags.items():
    s = 0.0
    for t in ts:
        s += float(idf_map.get(str(t), 0.0))
        tag_freq[t] = tag_freq.get(t, 0) + 1
    denom_idf[d] = s
tag_cov_docs = len(doc2tags)
print(f"[TAG] docs_with_tags={tag_cov_docs:,} / {N:,} ({tag_cov_docs/N:.3%}), idf_size={len(idf_map)}")

# ---------- 读取行为基表（Org/Creator 银标） ----------
beh = pd.read_parquet(TMP_DIR/"beh_base.parquet", engine=PARQUET_ENGINE)
org_arr = np.full(N, -1, dtype=np.int64)
cre_arr = np.full(N, -1, dtype=np.int64)
idx = beh["doc_idx"].to_numpy(np.int64)
if "OwnerOrganizationId" in beh.columns:
    orgv = beh["OwnerOrganizationId"].fillna(-1).astype("Int64").to_numpy()
else:
    orgv = np.full(idx.size, -1, dtype="Int64")
if "CreatorUserId" in beh.columns:
    crev = beh["CreatorUserId"].fillna(-1).astype("Int64").to_numpy()
else:
    crev = np.full(idx.size, -1, dtype="Int64")
org_arr[idx] = np.array(orgv, dtype=np.int64)
cre_arr[idx] = np.array(crev, dtype=np.int64)

# 组规模（用于 Recall 分母）
def group_sizes(arr: np.ndarray):
    # 仅统计 >=0 的 id
    ids, cnts = np.unique(arr[arr>=0], return_counts=True)
    return dict(zip(ids.tolist(), cnts.tolist()))
org_size = group_sizes(org_arr)
cre_size = group_sizes(cre_arr)

# ---------- 指标累加器 ----------
def zero_metrics():
    return dict(ndcg=0.0, ap=0.0, rr=0.0, p=0.0, r=0.0, covered=0, total=0)

m_tag = zero_metrics()
m_org = zero_metrics()
m_cre = zero_metrics()

# 预先计算 IDCG@K（上界：每位增益视为1）
IDCG = sum([1.0 / math.log2(r+2) for r in range(K_EVAL)])

def update_binary_metrics(m, rel_flags):
    """rel_flags: 长度<=K 的 0/1 numpy 向量（按排名）"""
    if rel_flags.size == 0:
        return
    K = rel_flags.size
    # P@K
    prec = rel_flags.sum() / float(K)
    # R@K: 需要在外部传入分母，外层处理
    # AP@K
    if rel_flags.sum() > 0:
        ranks = np.where(rel_flags>0)[0]  # 0-based ranks
        prec_at_hits = [(rel_flags[:(r+1)].sum() / float(r+1)) for r in ranks]
        ap = float(np.mean(prec_at_hits))
        rr = 1.0 / float(ranks[0]+1)
    else:
        ap = 0.0; rr = 0.0
    m["ap"] += ap
    m["rr"] += rr
    m["p"]  += prec

def update_ndcg(m, gains):
    """gains: 长度<=K 的 [0,1] 归一增益"""
    if gains.size == 0:
        return
    dcg = float(np.sum(gains / np.log2(np.arange(gains.size)+2)))
    m["ndcg"] += (dcg / IDCG)

def finalize(m):
    tot = max(m["total"], 1)
    return dict(
        nDCG = m["ndcg"]/tot,
        MAP  = m["ap"]/tot,
        MRR  = m["rr"]/tot,
        P    = m["p"]/tot,
        R    = m["r"]/tot,
        Coverage = m["covered"]/tot
    )

# ---------- 逐分片评测 ----------
for pi, fn in enumerate(files, 1):
    df = pd.read_parquet(TMP_DIR/fn, engine=PARQUET_ENGINE)
    # 期望列: row,col,val（本步已是 Top-50，且行内已归一）
    rows = df["row"].to_numpy(np.int64, copy=False)
    cols = df["col"].to_numpy(np.int64, copy=False)
    vals = df["val"].to_numpy(np.float32, copy=False)

    order = np.argsort(rows, kind="stable")
    rows, cols, vals = rows[order], cols[order], vals[order]
    uniq_rows, start = np.unique(rows, return_index=True)
    start = np.r_[start, len(rows)]

    for k in range(len(uniq_rows)):
        r = int(uniq_rows[k]); s, e = int(start[k]), int(start[k+1])
        # Top-K=20
        take = slice(s, min(s+K_EVAL, e))
        nbrs = cols[take]
        # ------- Tag -------
        tags_r = doc2tags.get(r, set())
        if tags_r:
            # 归一化增益（基于 IDF）
            denom = float(denom_idf[r]) + 1e-8
            gains = np.zeros(nbrs.size, dtype=np.float32)
            flags = np.zeros(nbrs.size, dtype=np.int32)
            for i, j in enumerate(nbrs):
                tj = doc2tags.get(int(j), set())
                if not tj: continue
                inter = tags_r.intersection(tj)
                if not inter: continue
                flags[i] = 1
                s_idf = 0.0
                for t in inter:
                    s_idf += float(idf_map.get(str(t), 0.0))
                gains[i] = min(1.0, s_idf/denom)
            # recall 分母（上界近似）
            denom_rec = 0
            for t in tags_r:
                denom_rec += max(0, tag_freq.get(t,0)-1)
            # 累加
            m_tag["covered"] += 1
            m_tag["total"]   += 1
            update_ndcg(m_tag, gains)
            update_binary_metrics(m_tag, flags)
            if denom_rec > 0:
                m_tag["r"] += float(flags.sum())/float(denom_rec)
        else:
            m_tag["total"] += 1  # 计入 coverage 分母

        # ------- Org -------
        org_r = org_arr[r]
        if org_r >= 0:
            flags = (org_arr[nbrs] == org_r).astype(np.int32)
            m_org["covered"] += 1
            m_org["total"]   += 1
            update_binary_metrics(m_org, flags)
            denom = max(1, org_size.get(int(org_r), 0) - 1)
            m_org["r"] += float(flags.sum())/float(denom)
            # nDCG：二值增益
            update_ndcg(m_org, flags.astype(np.float32))
        else:
            m_org["total"] += 1

        # ------- Creator -------
        cre_r = cre_arr[r]
        if cre_r >= 0:
            flags = (cre_arr[nbrs] == cre_r).astype(np.int32)
            m_cre["covered"] += 1
            m_cre["total"]   += 1
            update_binary_metrics(m_cre, flags)
            denom = max(1, cre_size.get(int(cre_r), 0) - 1)
            m_cre["r"] += float(flags.sum())/float(denom)
            update_ndcg(m_cre, flags.astype(np.float32))
        else:
            m_cre["total"] += 1

    if (pi % 4) == 0 or pi == len(files):
        print(f"[EVAL] processed {pi}/{len(files)} parts")

# ---------- 汇总并写表 ----------
res_tag = finalize(m_tag)
res_org = finalize(m_org)
res_cre = finalize(m_cre)

row = {
    "method": "Fused3-RR",
    # Tag
    "Tag-nDCG@20": res_tag["nDCG"], "Tag-MAP@20": res_tag["MAP"], "Tag-MRR@20": res_tag["MRR"],
    "Tag-P@20": res_tag["P"], "Tag-R@20": res_tag["R"], "Tag-Coverage": res_tag["Coverage"],
    # Org
    "Org-nDCG@20": res_org["nDCG"], "Org-MAP@20": res_org["MAP"], "Org-MRR@20": res_org["MRR"],
    "Org-P@20": res_org["P"], "Org-R@20": res_org["R"], "Org-Coverage": res_org["Coverage"],
    # Creator
    "Creator-nDCG@20": res_cre["nDCG"], "Creator-MAP@20": res_cre["MAP"], "Creator-MRR@20": res_cre["MRR"],
    "Creator-P@20": res_cre["P"], "Creator-R@20": res_cre["R"], "Creator-Coverage": res_cre["Coverage"],
}

# 统一指标（加权）
row["Unified@nDCG20"]      = W_TAG*row["Tag-nDCG@20"] + W_ORG*row["Org-nDCG@20"] + W_CRE*row["Creator-nDCG@20"]
row["Unified_cov@nDCG20"]  = W_TAG*row["Tag-Coverage"] + W_ORG*row["Org-Coverage"] + W_CRE*row["Creator-Coverage"]

out_df = pd.DataFrame([row])
print("\n[RESULT] Fused3-RR @K=20")
display(out_df)

# 追加写入 csv
csv_path = TMP_DIR/"metrics_main.csv"
if csv_path.exists():
    base = pd.read_csv(csv_path)
    base = pd.concat([base, out_df], ignore_index=True)
    base.to_csv(csv_path, index=False)
else:
    out_df.to_csv(csv_path, index=False)
print(f"[SAVE] appended to {csv_path}")


[Env] TMP_DIR=/Users/koyo/workspace/recsys/tmp
[MAN] S_fused3_rr_k50: nodes=521,735, parts=53
[TAG] docs_with_tags=521,735 / 521,735 (100.000%), idf_size=394
[EVAL] processed 4/53 parts
[EVAL] processed 8/53 parts
[EVAL] processed 12/53 parts
[EVAL] processed 16/53 parts
[EVAL] processed 20/53 parts
[EVAL] processed 24/53 parts
[EVAL] processed 28/53 parts
[EVAL] processed 32/53 parts
[EVAL] processed 36/53 parts
[EVAL] processed 40/53 parts
[EVAL] processed 44/53 parts
[EVAL] processed 48/53 parts
[EVAL] processed 52/53 parts
[EVAL] processed 53/53 parts

[RESULT] Fused3-RR @K=20


,method,Tag-nDCG@20,Tag-MAP@20,Tag-MRR@20,Tag-P@20,Tag-R@20,Tag-Coverage,Org-nDCG@20,Org-MAP@20,Org-MRR@20,...,Org-R@20,Org-Coverage,Creator-nDCG@20,Creator-MAP@20,Creator-MRR@20,Creator-P@20,Creator-R@20,Creator-Coverage,Unified@nDCG20,Unified_cov@nDCG20
0,Fused3-RR,0.088108,0.187412,0.191586,0.127827,0.000203,0.411291,0.003627,0.004318,0.004384,...,0.001829,0.004943,0.39409,0.734146,0.73949,0.319931,0.622271,1.0,0.171455,0.547269


[SAVE] appended to /Users/koyo/workspace/recsys/tmp/metrics_main.csv


In [5]:
# =========================================
# Step RR-BLEND-1 (fixed v2) · Build S_fused3_blend_eta020_k50
# 读:  tmp/S_fused3_symrow_k50_manifest.json + 分片
#     tmp/S_fused3_rr_k50_manifest.json + 分片
# 写:  tmp/S_fused3_blend_eta020_k50_partXXXX.parquet + manifest
# =========================================
import json
from pathlib import Path
import numpy as np
import pandas as pd

PARQUET_ENGINE = "fastparquet"
TMP_DIR = Path("./tmp").resolve()

PREF_RA = "S_fused3_symrow_k50"   # 行自适应融合（RA）
PREF_RR = "S_fused3_rr_k50"       # rerank 后的图（RR）

ETA = 0.20                 # 融合权重: 20% RR + 80% RA
TOPK = 50
FLUSH_EDGES = 2_000_000    # 每分片写出边数（近似）
OUT_PREF = f"S_fused3_blend_eta{int(ETA*100):03d}_k{TOPK:02d}"

print(f"[Env] TMP_DIR={TMP_DIR}")
print(f"[CFG] Blend: (1-eta)*RA + eta*RR, eta={ETA}, K={TOPK}")

def _read_manifest(prefix: str):
    """兼容多种 manifest 结构：
       - {"nodes":int, "parts":int, "nnz":int, "files":[...]}  (推荐)
       - {"nodes":int, "parts":[file,...], "nnz":int}          (parts 直接是文件名列表)
       - {"nodes":int, "parts":int}                            (无 files，用 parts 数构造默认文件名)
    """
    mp = TMP_DIR / f"{prefix}_manifest.json"
    with open(mp, "r") as f:
        man = json.load(f)

    nodes = man.get("nodes") or man.get("N") or man.get("docs")
    if nodes is None:
        raise ValueError(f"[FATAL] manifest {mp} 缺少 'nodes'")
    nodes = int(nodes)

    files = man.get("files")
    if not files:
        parts_field = man.get("parts")
        if isinstance(parts_field, list):
            files = parts_field
        elif isinstance(parts_field, (int, float, str)):
            parts_int = int(parts_field)
            files = [f"{prefix}_part{p:04d}.parquet" for p in range(parts_int)]
        else:
            files = sorted([p.name for p in TMP_DIR.glob(f"{prefix}_part*.parquet")])
            if not files:
                raise ValueError(f"[FATAL] 无法从 manifest 推断分片文件: {mp}")

    nnz = man.get("nnz") or man.get("total_edges")
    if isinstance(nnz, list):
        nnz = int(sum(int(x) for x in nnz))
    elif nnz is not None:
        nnz = int(nnz)

    return nodes, files, nnz

def _csr_from_parts(prefix: str):
    """两遍扫描将行随机图加载成 CSR（indptr, indices, values）"""
    N, files, _ = _read_manifest(prefix)

    # pass1: 行计数
    counts = np.zeros(N, dtype=np.int64)
    for i, fn in enumerate(files, 1):
        df = pd.read_parquet(TMP_DIR / fn, engine=PARQUET_ENGINE)
        r = df["row"].to_numpy(np.int64, copy=False)
        uniq, cnt = np.unique(r, return_counts=True)
        counts[uniq] += cnt
        if (i % 4) == 0 or i == len(files):
            print(f"[PASS1] {prefix} parts {i}/{len(files)} counted")

    indptr = np.zeros(N + 1, dtype=np.int64)
    np.cumsum(counts, out=indptr[1:])
    nnz = int(indptr[-1])

    indices = np.empty(nnz, dtype=np.int32)
    values  = np.empty(nnz, dtype=np.float32)
    cursor  = indptr.copy()

    # pass2: 填充
    for i, fn in enumerate(files, 1):
        df = pd.read_parquet(TMP_DIR / fn, engine=PARQUET_ENGINE)
        r = df["row"].to_numpy(np.int64, copy=False)
        c = df["col"].to_numpy(np.int64, copy=False).astype(np.int32, copy=False)
        v = df["val"].to_numpy(np.float32, copy=False)

        order = np.argsort(r, kind="stable")
        r, c, v = r[order], c[order], v[order]

        uniq, start = np.unique(r, return_index=True)
        start = np.r_[start, len(r)]
        for k in range(len(uniq)):
            rr = int(uniq[k]); s, e = int(start[k]), int(start[k + 1])
            pos = cursor[rr]; n = e - s
            indices[pos:pos + n] = c[s:e]
            values[pos:pos + n]  = v[s:e]
            cursor[rr] += n

        if (i % 4) == 0 or i == len(files):
            print(f"[PASS2] {prefix} parts {i}/{len(files)} filled")

    return N, indptr, indices, values

# ---------- 加载 RA 与 RR ----------
N_ra, ip_ra, ix_ra, vx_ra = _csr_from_parts(PREF_RA)
N_rr, ip_rr, ix_rr, vx_rr = _csr_from_parts(PREF_RR)
assert N_ra == N_rr, "[FATAL] N mismatch"
N = N_ra
print(f"[CSR] loaded both graphs. N={N:,}, RA_nnz={ip_ra[-1]:,}, RR_nnz={ip_rr[-1]:,}")

# ---------- 行融合 + TopK + 归一 ----------
out_rows, out_cols, out_vals = [], [], []
edges_in_buffer, part_idx = 0, 0

def _flush():
    """把缓冲边写出为一个分片，并清空缓冲。"""
    global out_rows, out_cols, out_vals, edges_in_buffer, part_idx
    if not out_rows:
        return
    df = pd.DataFrame({
        "row": np.asarray(out_rows, dtype=np.int64),
        "col": np.asarray(out_cols, dtype=np.int64),
        "val": np.asarray(out_vals, dtype=np.float32),
    })
    fn = f"{OUT_PREF}_part{part_idx:04d}.parquet"
    df.to_parquet(TMP_DIR / fn, engine=PARQUET_ENGINE, index=False)
    print(f"[SAVE] {fn} edges={len(df)}")
    part_idx += 1
    out_rows.clear(); out_cols.clear(); out_vals.clear()
    edges_in_buffer = 0

for r in range(N):
    s1, e1 = ip_ra[r], ip_ra[r+1]
    s2, e2 = ip_rr[r], ip_rr[r+1]

    pool = {}
    if e1 > s1:
        for p in range(s1, e1):
            j = int(ix_ra[p])
            pool[j] = pool.get(j, 0.0) + (1.0 - ETA) * float(vx_ra[p])
    if e2 > s2:
        for p in range(s2, e2):
            j = int(ix_rr[p])
            pool[j] = pool.get(j, 0.0) + ETA * float(vx_rr[p])

    if not pool:
        continue

    items = list(pool.items())
    if len(items) > TOPK:
        sel = np.argpartition([-w for _, w in items], TOPK-1)[:TOPK]
        items = [items[i] for i in sel]

    cols = np.array([j for j, _ in items], dtype=np.int64)
    vals = np.array([w for _, w in items], dtype=np.float32)
    s = float(vals.sum())
    if s > 0:
        vals /= s
    order = np.argsort(-vals, kind="stable")
    cols, vals = cols[order], vals[order]

    out_rows.extend([r]*len(cols))
    out_cols.extend(cols.tolist())
    out_vals.extend(vals.tolist())
    edges_in_buffer += len(cols)

    if edges_in_buffer >= FLUSH_EDGES:
        _flush()

    if (r+1) % 200_000 == 0:
        print(f"[PROG] rows processed {r+1:,}/{N:,}")

_flush()

# ---------- 写 manifest ----------
part_files = [f"{OUT_PREF}_part{p:04d}.parquet" for p in range(part_idx)]
nnz_sum = 0
for fn in part_files:
    nnz_sum += pd.read_parquet(TMP_DIR / fn, engine=PARQUET_ENGINE).shape[0]

man = {"nodes": int(N), "parts": part_files, "nnz": int(nnz_sum), "files": part_files}
with open(TMP_DIR / f"{OUT_PREF}_manifest.json", "w") as f:
    json.dump(man, f)

print(f"[MANIFEST] {OUT_PREF}_manifest.json  parts={len(part_files)}, nnz≈{nnz_sum:,}")
print("[Step RR-BLEND-1] DONE: blended graph generated.")


[Env] TMP_DIR=/Users/koyo/workspace/recsys/tmp
[CFG] Blend: (1-eta)*RA + eta*RR, eta=0.2, K=50
[PASS1] S_fused3_symrow_k50 parts 4/14 counted
[PASS1] S_fused3_symrow_k50 parts 8/14 counted
[PASS1] S_fused3_symrow_k50 parts 12/14 counted
[PASS1] S_fused3_symrow_k50 parts 14/14 counted
[PASS2] S_fused3_symrow_k50 parts 4/14 filled
[PASS2] S_fused3_symrow_k50 parts 8/14 filled
[PASS2] S_fused3_symrow_k50 parts 12/14 filled
[PASS2] S_fused3_symrow_k50 parts 14/14 filled
[PASS1] S_fused3_rr_k50 parts 4/53 counted
[PASS1] S_fused3_rr_k50 parts 8/53 counted
[PASS1] S_fused3_rr_k50 parts 12/53 counted
[PASS1] S_fused3_rr_k50 parts 16/53 counted
[PASS1] S_fused3_rr_k50 parts 20/53 counted
[PASS1] S_fused3_rr_k50 parts 24/53 counted
[PASS1] S_fused3_rr_k50 parts 28/53 counted
[PASS1] S_fused3_rr_k50 parts 32/53 counted
[PASS1] S_fused3_rr_k50 parts 36/53 counted
[PASS1] S_fused3_rr_k50 parts 40/53 counted
[PASS1] S_fused3_rr_k50 parts 44/53 counted
[PASS1] S_fused3_rr_k50 parts 48/53 counted
[PA

In [6]:
# =========================================
# Step RR-BLEND-2 · 评测混合图 (η=0.20) 并追加到 metrics_main.csv
# 读:  tmp/S_fused3_blend_eta020_k50_manifest.json + 分片
#     tmp/relevance_tag_docs.parquet, tmp/relevance_tag_idf.parquet
#     tmp/relevance_org.parquet, tmp/relevance_creator.parquet
# 写:  tmp/metrics_main.csv （追加）
# =========================================
import json
from pathlib import Path
import numpy as np
import pandas as pd

PARQUET_ENGINE = "fastparquet"
TMP_DIR = Path("./tmp").resolve()

# ---- 评测设置 ----
GRAPH_PREF   = "S_fused3_blend_eta020_k50"
METHOD_NAME  = "Fused3-Blend-eta0.20"
K_EVAL       = 20

# 统一指标权重（已与你确认的 0.6 / 0.1 / 0.3）
W_TAG, W_ORG, W_CRE = 0.6, 0.1, 0.3

print(f"[Env] TMP_DIR={TMP_DIR}")
print(f"[CFG] Evaluate {METHOD_NAME} @K={K_EVAL}")

# ---------- 读取混合图 manifest ----------
with open(TMP_DIR / f"{GRAPH_PREF}_manifest.json","r") as f:
    man = json.load(f)
nodes = int(man.get("nodes"))
parts = man.get("files") or man.get("parts")
if isinstance(parts, int):
    part_files = [f"{GRAPH_PREF}_part{p:04d}.parquet" for p in range(parts)]
elif isinstance(parts, list):
    part_files = parts
else:
    part_files = sorted([p.name for p in TMP_DIR.glob(f"{GRAPH_PREF}_part*.parquet")])
print(f"[MAN] {GRAPH_PREF}: nodes={nodes:,}, parts={len(part_files)}")

# ---------- 读取银标 ----------
# Tag：每文档标签列表（list[str]），以及 tag 的 idf 可选（这里仅用于后续扩展；当前用二值相关一致口径）
tag_docs_df = pd.read_parquet(TMP_DIR / "relevance_tag_docs.parquet", engine=PARQUET_ENGINE)
N_docs = int(tag_docs_df["doc_idx"].max()) + 1
assert N_docs == nodes, f"[FATAL] node size mismatch: tags N={N_docs} vs graph N={nodes}"

# 将标签列表放入定长数组（缺失为空列表）
tag_lists = [None] * nodes
for r in tag_docs_df.itertuples(index=False):
    idx = int(r.doc_idx)
    # r.tags 可能是 list[str] 或逗号串，统一成 set[str]
    tags = r.tags
    if isinstance(tags, str):
        tags = [t.strip() for t in tags.split(",") if t.strip()]
    tag_lists[idx] = set(tags) if tags else set()
tag_docs_df = None  # 释放

# 构建 tag 倒排（用于估计每个查询的 N_rel 以求 IDCG）
tag2docs = {}
for i, s in enumerate(tag_lists):
    if not s:
        continue
    for t in s:
        tag2docs.setdefault(t, []).append(i)
for t in list(tag2docs.keys()):
    tag2docs[t] = np.asarray(tag2docs[t], dtype=np.int64)

# Org / Creator：单一标签（-1 表示缺失）
org_df = pd.read_parquet(TMP_DIR / "relevance_org.parquet", engine=PARQUET_ENGINE)
creator_df = pd.read_parquet(TMP_DIR / "relevance_creator.parquet", engine=PARQUET_ENGINE)

org_arr = np.full(nodes, -1, dtype=np.int64)
org_arr[org_df["doc_idx"].to_numpy(np.int64)] = org_df["org_id"].to_numpy(np.int64)
creator_arr = np.full(nodes, -1, dtype=np.int64)
creator_arr[creator_df["doc_idx"].to_numpy(np.int64)] = creator_df["creator_id"].to_numpy(np.int64)

# 统计每个 org / creator 的文档数（用于 N_rel）
org_counts = pd.Series(org_arr).value_counts().to_dict()
creator_counts = pd.Series(creator_arr).value_counts().to_dict()

# ---------- 指标工具 ----------
LOG2 = np.log(2.0)
def dcg_at_k(rel):
    # rel 为 0/1 的 ndarray，长度 <= K
    if rel.size == 0:
        return 0.0
    idx = np.arange(1, rel.size + 1, dtype=np.float64)
    return np.sum(rel / np.log2(idx + 1.0))

def idcg_at_k(n_rel, k):
    m = int(min(n_rel, k))
    if m <= 0:
        return 0.0
    idx = np.arange(1, m + 1, dtype=np.float64)
    return np.sum(1.0 / np.log2(idx + 1.0))

def ap_at_k(rel):
    # 0/1 relevance，AP@K = Σ precision@i (在每个命中位置 i) / min(K, N_rel)
    if rel.size == 0:
        return 0.0
    hits = np.where(rel == 1)[0]
    if hits.size == 0:
        return 0.0
    precs = [(i+1) / (hit+1) for i, hit in enumerate(hits)]
    denom = max(1, min(rel.size, int(rel.sum())))  # min(K, N_rel)
    return float(np.sum(precs) / denom)

def rr_at_k(rel):
    hits = np.where(rel == 1)[0]
    return float(1.0/(hits[0]+1)) if hits.size > 0 else 0.0

# ---------- 评测累加器 ----------
# 对每一种任务，仅在“可评估”的行内取均值，并单独给出 Coverage（可评估行 / 全体）
tag_cov_cnt = 0
tag_ndcg_sum = tag_map_sum = tag_mrr_sum = tag_p_sum = tag_r_sum = 0.0

org_cov_cnt = 0
org_ndcg_sum = org_map_sum = org_mrr_sum = org_p_sum = org_r_sum = 0.0

creator_cov_cnt = 0
creator_ndcg_sum = creator_map_sum = creator_mrr_sum = creator_p_sum = creator_r_sum = 0.0

def eval_one_row(q, neigh):
    """neigh: ndarray[int] 预测的 top-K 邻居（不含排序权）"""
    global tag_cov_cnt, tag_ndcg_sum, tag_map_sum, tag_mrr_sum, tag_p_sum, tag_r_sum
    global org_cov_cnt, org_ndcg_sum, org_map_sum, org_mrr_sum, org_p_sum, org_r_sum
    global creator_cov_cnt, creator_ndcg_sum, creator_map_sum, creator_mrr_sum, creator_p_sum, creator_r_sum

    K = min(K_EVAL, neigh.size)
    if K == 0:
        return
    neigh = neigh[:K]

    # -------- Tag 评测（二值相关：是否有共享标签；IDCG 依据查询的真实 N_rel）
    q_tags = tag_lists[q]
    if q_tags and len(q_tags) > 0:
        tag_cov_cnt += 1
        # 计算 N_rel（全集中与 q 共享任一 tag 的文档数，排除自身）
        cand_union = None
        for t in q_tags:
            arr = tag2docs.get(t)
            if arr is None: 
                continue
            cand_union = arr if cand_union is None else np.union1d(cand_union, arr)
        if cand_union is None:
            n_rel = 0
        else:
            n_rel = int(cand_union.size - (1 if q in set(cand_union.tolist()) else 0))

        # 邻居二值相关
        rel = np.zeros(K, dtype=np.int8)
        if len(q_tags) > 0:
            # 更高效：把 q_tags 转为局部 set，逐一判断
            qset = q_tags
            for i, j in enumerate(neigh):
                tj = tag_lists[int(j)]
                rel[i] = 1 if (tj and len(qset.intersection(tj)) > 0) else 0

        # nDCG / MAP / MRR / P / R
        nd = 0.0
        idc = idcg_at_k(n_rel, K_EVAL)
        if idc > 0:
            nd = dcg_at_k(rel) / idc
        tag_ndcg_sum += nd
        tag_map_sum  += ap_at_k(rel)
        tag_mrr_sum  += rr_at_k(rel)
        tag_p_sum    += float(rel.sum()) / K_EVAL
        tag_r_sum    += (float(rel.sum()) / max(1, n_rel)) if n_rel > 0 else 0.0

    # -------- Org 评测（同 Org）
    q_org = int(org_arr[q])
    if q_org != -1 and org_counts.get(q_org, 0) > 1:
        org_cov_cnt += 1
        n_rel = org_counts[q_org] - 1
        rel = (org_arr[neigh] == q_org).astype(np.int8, copy=False)

        idc = idcg_at_k(n_rel, K_EVAL)
        nd  = (dcg_at_k(rel) / idc) if idc > 0 else 0.0
        org_ndcg_sum += nd
        org_map_sum  += ap_at_k(rel)
        org_mrr_sum  += rr_at_k(rel)
        org_p_sum    += float(rel.sum()) / K_EVAL
        org_r_sum    += (float(rel.sum()) / max(1, n_rel)) if n_rel > 0 else 0.0

    # -------- Creator 评测（同 Creator）
    q_cr = int(creator_arr[q])
    if q_cr != -1 and creator_counts.get(q_cr, 0) > 1:
        creator_cov_cnt += 1
        n_rel = creator_counts[q_cr] - 1
        rel = (creator_arr[neigh] == q_cr).astype(np.int8, copy=False)

        idc = idcg_at_k(n_rel, K_EVAL)
        nd  = (dcg_at_k(rel) / idc) if idc > 0 else 0.0
        creator_ndcg_sum += nd
        creator_map_sum  += ap_at_k(rel)
        creator_mrr_sum  += rr_at_k(rel)
        creator_p_sum    += float(rel.sum()) / K_EVAL
        creator_r_sum    += (float(rel.sum()) / max(1, n_rel)) if n_rel > 0 else 0.0

# ---------- 流式评测 ----------
ROWS_PER_PRINT = 200_000
rows_done = 0

for pi, fn in enumerate(part_files, 1):
    df = pd.read_parquet(TMP_DIR / fn, engine=PARQUET_ENGINE)
    # 假定每分片内已按 row 聚合；保险起见还是 groupby
    for r, grp in df.groupby("row"):
        neigh = grp.sort_values("val", ascending=False)["col"].to_numpy(np.int64, copy=False)
        eval_one_row(int(r), neigh)
        rows_done += 1
        if (rows_done % ROWS_PER_PRINT) == 0:
            print(f"[EVAL] processed {rows_done:,}/{nodes:,} rows")
    print(f"[PART] {pi}/{len(part_files)} done")

# ---------- 汇总 & 写出 ----------
eps = 1e-12
tag_cov_rate     = tag_cov_cnt / nodes
org_cov_rate     = org_cov_cnt / nodes
creator_cov_rate = creator_cov_cnt / nodes

def safe_mean(s, n): 
    return (s / max(1, n))

row = {
    "method": METHOD_NAME,
    "Tag-nDCG@20":     safe_mean(tag_ndcg_sum,     tag_cov_cnt),
    "Tag-MAP@20":      safe_mean(tag_map_sum,      tag_cov_cnt),
    "Tag-MRR@20":      safe_mean(tag_mrr_sum,      tag_cov_cnt),
    "Tag-P@20":        safe_mean(tag_p_sum,        tag_cov_cnt),
    "Tag-R@20":        safe_mean(tag_r_sum,        tag_cov_cnt),
    "Tag-Coverage":    tag_cov_rate,

    "Org-nDCG@20":     safe_mean(org_ndcg_sum,     org_cov_cnt),
    "Org-MAP@20":      safe_mean(org_map_sum,      org_cov_cnt),
    "Org-MRR@20":      safe_mean(org_mrr_sum,      org_cov_cnt),
    "Org-P@20":        safe_mean(org_p_sum,        org_cov_cnt),
    "Org-R@20":        safe_mean(org_r_sum,        org_cov_cnt),
    "Org-Coverage":    org_cov_rate,

    "Creator-nDCG@20": safe_mean(creator_ndcg_sum, creator_cov_cnt),
    "Creator-MAP@20":  safe_mean(creator_map_sum,  creator_cov_cnt),
    "Creator-MRR@20":  safe_mean(creator_mrr_sum,  creator_cov_cnt),
    "Creator-P@20":    safe_mean(creator_p_sum,    creator_cov_cnt),
    "Creator-R@20":    safe_mean(creator_r_sum,    creator_cov_cnt),
    "Creator-Coverage":creator_cov_rate,
}

# 统一指标（按你指定的 0.6/0.1/0.3）
row["Unified@nDCG20"]      = W_TAG*row["Tag-nDCG@20"] + W_ORG*row["Org-nDCG@20"] + W_CRE*row["Creator-nDCG@20"]
row["Unified_cov@nDCG20"]  = W_TAG*(row["Tag-nDCG@20"]*row["Tag-Coverage"]) \
                           + W_ORG*(row["Org-nDCG@20"]*row["Org-Coverage"]) \
                           + W_CRE*(row["Creator-nDCG@20"]*row["Creator-Coverage"])

out_df = pd.DataFrame([row])
csv_path = TMP_DIR / "metrics_main.csv"
if csv_path.exists():
    base = pd.read_csv(csv_path)
    base = pd.concat([base, out_df], axis=0, ignore_index=True)
    base.to_csv(csv_path, index=False)
else:
    out_df.to_csv(csv_path, index=False)

display(out_df.round(6))
print(f"[SAVE] appended to {csv_path}")


[Env] TMP_DIR=/Users/koyo/workspace/recsys/tmp
[CFG] Evaluate Fused3-Blend-eta0.20 @K=20
[MAN] S_fused3_blend_eta020_k50: nodes=521,735, parts=14
[PART] 1/14 done
[PART] 2/14 done
[PART] 3/14 done
[PART] 4/14 done
[EVAL] processed 200,000/521,735 rows
[PART] 5/14 done
[PART] 6/14 done
[PART] 7/14 done
[PART] 8/14 done
[PART] 9/14 done
[EVAL] processed 400,000/521,735 rows
[PART] 10/14 done
[PART] 11/14 done
[PART] 12/14 done
[PART] 13/14 done
[PART] 14/14 done


,method,Tag-nDCG@20,Tag-MAP@20,Tag-MRR@20,Tag-P@20,Tag-R@20,Tag-Coverage,Org-nDCG@20,Org-MAP@20,Org-MRR@20,...,Org-R@20,Org-Coverage,Creator-nDCG@20,Creator-MAP@20,Creator-MRR@20,Creator-P@20,Creator-R@20,Creator-Coverage,Unified@nDCG20,Unified_cov@nDCG20
0,Fused3-Blend-eta0.20,0.197454,0.360188,0.367804,0.170117,0.000342,0.411291,0.693488,0.742792,0.718254,...,0.366986,0.004504,0.890474,0.896391,0.877651,0.375943,0.799523,0.7759,0.454963,0.256315


[SAVE] appended to /Users/koyo/workspace/recsys/tmp/metrics_main.csv


In [7]:
# =========================================
# Step RR-BLEND-GRID · η 网格融合 + 评测（一次跑全）
# 读:  tmp/S_fused3_symrow_k50_*.parquet + manifest (RA)
#     tmp/S_fused3_rr_k50_*.parquet      + manifest (RR)
#     tmp/relevance_tag_docs.parquet / relevance_org.parquet / relevance_creator.parquet
# 写:  tmp/S_fused3_blend_etaXXX_k50_part*.parquet + manifest
#     tmp/metrics_main.csv (追加)
# =========================================
import json, math, gc
from pathlib import Path
import numpy as np
import pandas as pd

PARQUET_ENGINE = "fastparquet"
TMP_DIR = Path("./tmp").resolve()
ETAS = [0.10, 0.15, 0.20, 0.25, 0.30]
K_KEEP = 50
K_EVAL = 20
W_TAG, W_ORG, W_CRE = 0.6, 0.1, 0.3

print(f"[Env] TMP_DIR={TMP_DIR}")

# ---------- 基础工具 ----------
def _read_manifest(prefix: str):
    man = json.load(open(TMP_DIR / f"{prefix}_manifest.json","r"))
    nodes = int(man["nodes"])
    files = man.get("files")
    if files is None:
        parts = man.get("parts")
        if isinstance(parts, int):
            files = [f"{prefix}_part{p:04d}.parquet" for p in range(parts)]
        else:
            files = sorted([p.name for p in TMP_DIR.glob(f"{prefix}_part*.parquet")])
    return nodes, files

def _iter_rows(prefix: str):
    """逐分片读取 (row,col,val)，yield (row -> [col,val] 排序后数组) 的迭代器"""
    nodes, files = _read_manifest(prefix)
    for fn in files:
        df = pd.read_parquet(TMP_DIR / fn, engine=PARQUET_ENGINE)
        for r, grp in df.groupby("row"):
            g = grp.sort_values("val", ascending=False)
            yield int(r), g["col"].to_numpy(np.int64, copy=False), g["val"].to_numpy(np.float32, copy=False)
        del df
        gc.collect()
    return

def _blend_and_save(eta: float, pref_ra: str, pref_rr: str, out_pref: str):
    """(1-eta)*RA + eta*RR，行内取 top-K_KEEP，流式写分片 + manifest"""
    nodes_ra, files_ra = _read_manifest(pref_ra)
    nodes_rr, files_rr = _read_manifest(pref_rr)
    assert nodes_ra == nodes_rr, "[FATAL] node mismatch RA vs RR"
    N = nodes_ra
    print(f"[BLEND] eta={eta:.2f} | N={N:,} | K={K_KEEP}")

    # 逐行双指针融合（两来源已各自按 val 降序）
    # 为避免大量内存占用，我们按若干万行 flush 到磁盘
    BUF_ROWS = []
    CHUNK_ROWS = 200_000
    part_idx = 0
    rows_seen = 0

    # 为加速：将 RR 分片建一个 row->(cols,vals) 的临时映射（分批），避免全量驻留
    # 简化处理：我们同步遍历两份分片生成的迭代器，以行为键做归并
    it_ra = _iter_rows(pref_ra)
    it_rr = _iter_rows(pref_rr)
    cur_ra = next(it_ra, None)
    cur_rr = next(it_rr, None)

    def flush():
        nonlocal part_idx, BUF_ROWS
        if not BUF_ROWS:
            return
        out_df = pd.concat(BUF_ROWS, axis=0, ignore_index=True)
        out_df.to_parquet(TMP_DIR / f"{out_pref}_part{part_idx:04d}.parquet",
                          engine=PARQUET_ENGINE, index=False)
        print(f"[SAVE] {out_pref}_part{part_idx:04d}.parquet rows={len(out_df):,}")
        part_idx += 1
        BUF_ROWS = []

    while cur_ra is not None or cur_rr is not None:
        if cur_ra is None:
            r = cur_rr[0]
        elif cur_rr is None:
            r = cur_ra[0]
        else:
            r = min(cur_ra[0], cur_rr[0])

        cols = []
        vals = []

        if cur_ra is not None and cur_ra[0] == r:
            _, c_ra, v_ra = cur_ra
            cols.append(c_ra); vals.append((1.0-eta)*v_ra)
            cur_ra = next(it_ra, None)
        if cur_rr is not None and cur_rr[0] == r:
            _, c_rr, v_rr = cur_rr
            cols.append(c_rr); vals.append(eta*v_rr)
            cur_rr = next(it_rr, None)

        if cols:
            C = np.concatenate(cols)
            V = np.concatenate(vals)
            # 合并重复 col
            order = np.argsort(C, kind="mergesort")
            C = C[order]; V = V[order]
            # reduce by key
            uniq, starts = np.unique(C, return_index=True)
            sums = np.add.reduceat(V, starts)
            # 降序取前 K
            top = np.argsort(-sums)[:K_KEEP]
            C_top = uniq[top]; V_top = sums[top]
            # 写入缓冲
            BUF_ROWS.append(pd.DataFrame({
                "row": np.full(C_top.size, r, dtype=np.int64),
                "col": C_top.astype(np.int64, copy=False),
                "val": V_top.astype(np.float32, copy=False),
            }))
        rows_seen += 1
        if rows_seen % CHUNK_ROWS == 0:
            flush()
            print(f"[PROG] blended rows={rows_seen:,}/{N:,}")

    # 最后一批
    flush()
    # 写 manifest
    man = {"nodes": N, "parts": part_idx}
    json.dump(man, open(TMP_DIR / f"{out_pref}_manifest.json","w"))
    print(f"[MANIFEST] {out_pref}_manifest.json parts={part_idx}")

# ---------- 评测工具（与前一步一致口径） ----------
LOG2 = np.log(2.0)
def dcg_at_k(rel):
    if rel.size == 0: return 0.0
    idx = np.arange(1, rel.size + 1, dtype=np.float64)
    return float(np.sum(rel / np.log2(idx + 1.0)))

def idcg_at_k(n_rel, k):
    m = int(min(n_rel, k))
    if m <= 0: return 0.0
    idx = np.arange(1, m + 1, dtype=np.float64)
    return float(np.sum(1.0 / np.log2(idx + 1.0)))

def ap_at_k(rel):
    if rel.size == 0: return 0.0
    hits = np.where(rel == 1)[0]
    if hits.size == 0: return 0.0
    precs = [(i+1) / (hit+1) for i, hit in enumerate(hits)]
    denom = max(1, min(rel.size, int(rel.sum())))
    return float(np.sum(precs) / denom)

def rr_at_k(rel):
    hits = np.where(rel == 1)[0]
    return float(1.0/(hits[0]+1)) if hits.size>0 else 0.0

def eval_graph(prefix: str, method_name: str):
    nodes, files = _read_manifest(prefix)

    # Tag 银标
    tag_docs_df = pd.read_parquet(TMP_DIR / "relevance_tag_docs.parquet", engine=PARQUET_ENGINE)
    assert int(tag_docs_df["doc_idx"].max())+1 == nodes
    tag_lists = [set() for _ in range(nodes)]
    for r in tag_docs_df.itertuples(index=False):
        tags = r.tags
        if isinstance(tags, str):
            tags = [t.strip() for t in tags.split(",") if t.strip()]
        tag_lists[int(r.doc_idx)] = set(tags) if tags else set()
    tag2docs = {}
    for i,s in enumerate(tag_lists):
        for t in s:
            tag2docs.setdefault(t, []).append(i)
    for t in list(tag2docs):
        tag2docs[t] = np.asarray(tag2docs[t], dtype=np.int64)

    # Org / Creator 银标
    org_df = pd.read_parquet(TMP_DIR / "relevance_org.parquet", engine=PARQUET_ENGINE)
    creator_df = pd.read_parquet(TMP_DIR / "relevance_creator.parquet", engine=PARQUET_ENGINE)
    org_arr = np.full(nodes, -1, dtype=np.int64)
    org_arr[org_df["doc_idx"].to_numpy(np.int64)] = org_df["org_id"].to_numpy(np.int64)
    creator_arr = np.full(nodes, -1, dtype=np.int64)
    creator_arr[creator_df["doc_idx"].to_numpy(np.int64)] = creator_df["creator_id"].to_numpy(np.int64)
    org_counts = pd.Series(org_arr).value_counts().to_dict()
    creator_counts = pd.Series(creator_arr).value_counts().to_dict()

    # 累加器
    tag_cov=org_cov=creator_cov = 0
    tag_nd=tag_ap=tag_rr=tag_p=tag_r = 0.0
    org_nd=org_ap=org_rr=org_p=org_r = 0.0
    cr_nd=cr_ap=cr_rr=cr_p=cr_r = 0.0

    rows_done=0
    for pi, fn in enumerate(files, 1):
        df = pd.read_parquet(TMP_DIR / fn, engine=PARQUET_ENGINE)
        for r, grp in df.groupby("row"):
            arr = grp.sort_values("val", ascending=False)["col"].to_numpy(np.int64, copy=False)[:K_EVAL]
            # Tag
            qtags = tag_lists[r]
            if qtags:
                tag_cov += 1
                # N_rel
                cand = None
                for t in qtags:
                    a = tag2docs.get(t)
                    cand = a if cand is None else np.union1d(cand, a)
                n_rel = 0 if cand is None else int(cand.size - (1 if r in set(cand.tolist()) else 0))
                rel = np.array([(len(tag_lists[j].intersection(qtags))>0) for j in arr], dtype=np.int8)
                idc = idcg_at_k(n_rel, K_EVAL)
                tag_nd += (dcg_at_k(rel)/idc) if idc>0 else 0.0
                tag_ap += ap_at_k(rel)
                tag_rr += rr_at_k(rel)
                tag_p  += float(rel.sum())/K_EVAL
                tag_r  += (float(rel.sum())/max(1,n_rel)) if n_rel>0 else 0.0

            # Org
            q_org = org_arr[r]
            if q_org != -1 and org_counts.get(q_org,0)>1:
                org_cov += 1
                n_rel = org_counts[q_org]-1
                rel = (org_arr[arr]==q_org).astype(np.int8)
                idc = idcg_at_k(n_rel, K_EVAL)
                org_nd += (dcg_at_k(rel)/idc) if idc>0 else 0.0
                org_ap += ap_at_k(rel)
                org_rr += rr_at_k(rel)
                org_p  += float(rel.sum())/K_EVAL
                org_r  += (float(rel.sum())/max(1,n_rel)) if n_rel>0 else 0.0

            # Creator
            q_cr = creator_arr[r]
            if q_cr != -1 and creator_counts.get(q_cr,0)>1:
                creator_cov += 1
                n_rel = creator_counts[q_cr]-1
                rel = (creator_arr[arr]==q_cr).astype(np.int8)
                idc = idcg_at_k(n_rel, K_EVAL)
                cr_nd += (dcg_at_k(rel)/idc) if idc>0 else 0.0
                cr_ap += ap_at_k(rel)
                cr_rr += rr_at_k(rel)
                cr_p  += float(rel.sum())/K_EVAL
                cr_r  += (float(rel.sum())/max(1,n_rel)) if n_rel>0 else 0.0

            rows_done += 1
        print(f"[EVAL] {prefix} parts {pi}/{len(files)} processed")

    def mean(x, n): return x/max(1,n)
    row = {
        "method": method_name,
        "Tag-nDCG@20":     mean(tag_nd, tag_cov),
        "Tag-MAP@20":      mean(tag_ap, tag_cov),
        "Tag-MRR@20":      mean(tag_rr, tag_cov),
        "Tag-P@20":        mean(tag_p,  tag_cov),
        "Tag-R@20":        mean(tag_r,  tag_cov),
        "Tag-Coverage":    tag_cov / nodes,

        "Org-nDCG@20":     mean(org_nd, org_cov),
        "Org-MAP@20":      mean(org_ap, org_cov),
        "Org-MRR@20":      mean(org_rr, org_cov),
        "Org-P@20":        mean(org_p,  org_cov),
        "Org-R@20":        mean(org_r,  org_cov),
        "Org-Coverage":    org_cov / nodes,

        "Creator-nDCG@20": mean(cr_nd,  creator_cov),
        "Creator-MAP@20":  mean(cr_ap,  creator_cov),
        "Creator-MRR@20":  mean(cr_rr,  creator_cov),
        "Creator-P@20":    mean(cr_p,   creator_cov),
        "Creator-R@20":    mean(cr_r,   creator_cov),
        "Creator-Coverage":creator_cov / nodes,
    }
    row["Unified@nDCG20"]     = W_TAG*row["Tag-nDCG@20"] + W_ORG*row["Org-nDCG@20"] + W_CRE*row["Creator-nDCG@20"]
    row["Unified_cov@nDCG20"] = W_TAG*(row["Tag-nDCG@20"]*row["Tag-Coverage"]) \
                              + W_ORG*(row["Org-nDCG@20"]*row["Org-Coverage"]) \
                              + W_CRE*(row["Creator-nDCG@20"]*row["Creator-Coverage"])
    return pd.DataFrame([row])

# ---------- 主流程：对每个 η 融合+评测 ----------
pref_ra = "S_fused3_symrow_k50"
pref_rr = "S_fused3_rr_k50"
all_rows = []

for eta in ETAS:
    out_pref = f"S_fused3_blend_eta{int(round(eta*100)):03d}_k50"
    _blend_and_save(eta, pref_ra, pref_rr, out_pref)
    df = eval_graph(out_pref, f"Fused3-Blend-eta{eta:.2f}")
    all_rows.append(df)
    # 追加到 metrics_main.csv
    csv_path = TMP_DIR / "metrics_main.csv"
    if csv_path.exists():
        base = pd.read_csv(csv_path)
        base = pd.concat([base, df], axis=0, ignore_index=True)
        base.to_csv(csv_path, index=False)
    else:
        df.to_csv(csv_path, index=False)

# 汇总展示
summary = pd.concat(all_rows, axis=0, ignore_index=True)
display(summary.sort_values("Unified@nDCG20", ascending=False).round(6))
print("[DONE] blend grid finished & metrics appended.")


[Env] TMP_DIR=/Users/koyo/workspace/recsys/tmp
[BLEND] eta=0.10 | N=521,735 | K=50
[SAVE] S_fused3_blend_eta010_k50_part0000.parquet rows=10,000,000
[PROG] blended rows=200,000/521,735
[SAVE] S_fused3_blend_eta010_k50_part0001.parquet rows=10,000,000
[PROG] blended rows=400,000/521,735
[SAVE] S_fused3_blend_eta010_k50_part0002.parquet rows=6,086,750
[MANIFEST] S_fused3_blend_eta010_k50_manifest.json parts=3
[EVAL] S_fused3_blend_eta010_k50 parts 1/3 processed
[EVAL] S_fused3_blend_eta010_k50 parts 2/3 processed
[EVAL] S_fused3_blend_eta010_k50 parts 3/3 processed
[BLEND] eta=0.15 | N=521,735 | K=50
[SAVE] S_fused3_blend_eta015_k50_part0000.parquet rows=10,000,000
[PROG] blended rows=200,000/521,735
[SAVE] S_fused3_blend_eta015_k50_part0001.parquet rows=10,000,000
[PROG] blended rows=400,000/521,735
[SAVE] S_fused3_blend_eta015_k50_part0002.parquet rows=6,086,750
[MANIFEST] S_fused3_blend_eta015_k50_manifest.json parts=3
[EVAL] S_fused3_blend_eta015_k50 parts 1/3 processed
[EVAL] S_fuse

,method,Tag-nDCG@20,Tag-MAP@20,Tag-MRR@20,Tag-P@20,Tag-R@20,Tag-Coverage,Org-nDCG@20,Org-MAP@20,Org-MRR@20,...,Org-R@20,Org-Coverage,Creator-nDCG@20,Creator-MAP@20,Creator-MRR@20,Creator-P@20,Creator-R@20,Creator-Coverage,Unified@nDCG20,Unified_cov@nDCG20
4,Fused3-Blend-eta0.30,0.213326,0.373912,0.377555,0.185165,0.000356,0.411291,0.719696,0.776366,0.745488,...,0.370922,0.004504,0.894420,0.900423,0.882302,0.379246,0.800029,0.7759,0.468291,0.261162
3,Fused3-Blend-eta0.25,0.210140,0.369008,0.372595,0.182988,0.000353,0.411291,0.707074,0.758616,0.730155,...,0.369235,0.004504,0.892456,0.898209,0.879825,0.377643,0.799783,0.7759,0.464528,0.259913
2,Fused3-Blend-eta0.20,0.197412,0.360309,0.367793,0.170051,0.000341,0.411291,0.693488,0.742792,0.718254,...,0.366986,0.004504,0.890391,0.895995,0.877213,0.375898,0.799520,0.7759,0.454914,0.256285
1,Fused3-Blend-eta0.15,0.191324,0.355583,0.362587,0.164236,0.000334,0.411291,0.679836,0.727857,0.705578,...,0.364500,0.004504,0.887831,0.893298,0.872997,0.373956,0.799223,0.7759,0.449127,0.254180
0,Fused3-Blend-eta0.10,0.186179,0.348192,0.355937,0.160532,0.000329,0.411291,0.664227,0.712756,0.694489,...,0.361297,0.004504,0.883838,0.889059,0.866470,0.371101,0.798738,0.7759,0.443281,0.251974


[DONE] blend grid finished & metrics appended.


In [8]:
# === Replot grouped charts (Tag/Text/Fusion), highlight "my method" ===
# 约定：不使用 seaborn；每个指标单独一张图；不设置任何特定颜色

import os, re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --------- 找到你的 tmp 目录并读入 metrics_main.csv ----------
CANDIDATE_TMP_DIRS = [
    Path("/home/koyo/workspace/recsys/tmp"),
    Path("/Users/koyo/workspace/recsys/tmp"),
    Path("./tmp"),
]
TMP_DIR = None
for p in CANDIDATE_TMP_DIRS:
    if (p / "metrics_main.csv").exists():
        TMP_DIR = p
        break
if TMP_DIR is None:
    raise FileNotFoundError("找不到 metrics_main.csv，请确认它位于上述候选路径之一。")

METRICS_CSV = TMP_DIR / "metrics_main.csv"
df = pd.read_csv(METRICS_CSV)
assert "method" in df.columns, "metrics_main.csv 必须包含 'method' 列"

# --------- 分组规则 ----------
def which_group(method: str) -> str | None:
    name = method.lower()
    if name.startswith("tag-"):
        return "tag"
    if name.startswith("text-"):
        return "text"
    # 融合家族（我们的融合图 + 标准检索融合基线）
    if any(k in name for k in [
        "fused3", "fusion-rrf", "fusion-combsum", "combmnz", "blend", "rr", "ra"
    ]):
        return "fusion"
    return None  # 其他不绘制

df["group"] = df["method"].apply(which_group)
plot_df = df[df["group"].isin(["tag", "text", "fusion"])].copy()

# 你希望在每组里加粗标记的“我的方法”
MY_METHOD_BY_GROUP = {
    "tag":   "Tag-SGNS",
    "text":  "Text-SGNS",
    "fusion":"Fused3-Blend-eta0.30",  # 如需换成 Fused3-RA / Fused3-RR 等，改这里
}

# 数值指标列（去掉非数值与可能的Unnamed索引列）
numeric_cols = [c for c in plot_df.columns
                if c not in ("method","group") and np.issubdtype(plot_df[c].dtype, np.number)
                and not c.lower().startswith("unnamed")]

# 输出目录（与 tmp 并列，便于管理）
OUT_DIR = TMP_DIR / "figs_replot"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def safe_name(s: str) -> str:
    return re.sub(r"[^0-9a-zA-Z@._\-]+", "_", s)

def barh_group_plot(group: str, metric: str, gdf: pd.DataFrame) -> Path | None:
    g = gdf[["method", metric]].dropna().copy()
    if g.empty:
        return None
    # 升序便于把最好的放在顶部（后面反转 y 轴）
    g = g.sort_values(metric, ascending=True)

    my_name = MY_METHOD_BY_GROUP.get(group)
    tick_labels = []
    for m in g["method"]:
        if my_name and m.strip().lower() == my_name.lower():
            tick_labels.append(f"{m} (my method)")
        else:
            tick_labels.append(m)

    plt.figure(figsize=(10, max(2.5, 0.35 * len(g))))
    plt.barh(range(len(g)), g[metric].values)  # 使用默认颜色
    ax = plt.gca()
    ax.set_yticks(range(len(g)))
    ax.set_yticklabels(tick_labels)
    ax.set_xlabel(metric)
    ax.set_title(f"{group.upper()} — {metric}")
    # 将“我的方法”加粗
    for t in ax.get_yticklabels():
        if "(my method)" in t.get_text():
            t.set_fontweight("bold")
    # 最佳在上
    ax.invert_yaxis()
    plt.tight_layout()

    out_path = OUT_DIR / f"{group}_{safe_name(metric)}.png"
    plt.savefig(out_path, dpi=150)
    plt.close()
    return out_path

# --------- 逐组逐指标绘图 ----------
saved_paths = []
for group in ["tag", "text", "fusion"]:
    gdf = plot_df[plot_df["group"] == group]
    for metric in numeric_cols:
        p = barh_group_plot(group, metric, gdf)
        if p is not None:
            saved_paths.append(p)

print(f"[DONE] 生成 {len(saved_paths)} 张图，输出目录：{OUT_DIR}")
for p in saved_paths[:10]:
    print("  -", p.name)
if len(saved_paths) > 10:
    print(f"  ... 其余 {len(saved_paths)-10} 张略")


[DONE] 生成 56 张图，输出目录：/Users/koyo/workspace/recsys/tmp/figs_replot
  - tag_Tag-nDCG@20.png
  - tag_Tag-MAP@20.png
  - tag_Tag-MRR@20.png
  - tag_Tag-P@20.png
  - tag_Tag-R@20.png
  - tag_Tag-Coverage.png
  - tag_Org-nDCG@20.png
  - tag_Org-MAP@20.png
  - tag_Org-MRR@20.png
  - tag_Org-P@20.png
  ... 其余 46 张略
